# Using `QubitPlaceholder`s

In PyQuil, we typically use integers to identify qubits

In [1]:
import warnings
from pyquil._deprecation import PyQuilDeprecationWarning
# Hide pyQuil 4.22's notices that the QVM is deprecated in favour of pyQuil v5
# (the QVM still works; see 0_Jupyter_Pykernel_from_Container_image.ipynb).
warnings.filterwarnings("ignore", category=PyQuilDeprecationWarning)

from pyquil import Program
from pyquil.gates import CNOT, H
print(Program(H(0), CNOT(0, 1)))

H 0
CNOT 0 1



However, when running on real, near-term QPUs we care about what particular physical qubits our program will run on. In fact, we may want to run the same program on an assortment of different qubits. This is where using `QubitPlaceholder`s comes in.

In [2]:
from pyquil.quilatom import QubitPlaceholder
q0 = QubitPlaceholder()
q1 = QubitPlaceholder()
prog = Program(H(q0), CNOT(q0, q1))
print(prog)

H Placeholder(QubitPlaceholder(0x2B62F40))
CNOT Placeholder(QubitPlaceholder(0x2B62F40)) Placeholder(QubitPlaceholder(0x26E6110))



If you try to use this program directly, it will not work

In [3]:
try:
    print(prog.out())
except (RuntimeError, ValueError) as e:
    print(e)

ValueError: Qubit has not yet been resolved

Instead, you must explicitly map the placeholders to physical qubits. By default, the function `address_qubits` will address qubits from 0 to N.

In [4]:
from pyquil.quil import address_qubits
print(address_qubits(prog))

H 0
CNOT 0 1



The real power comes into play when you provide an explicit mapping

In [5]:
print(address_qubits(prog, qubit_mapping={
    q0: 14,
    q1: 19,
}))

H 14
CNOT 14 19



## Register
Usually, your algorithm will use an assortment of qubits. You can use the convenience function `QubitPlaceholder.register()` to request a list of qubits to build your program.

In [6]:
qbyte = QubitPlaceholder.register(8)
prog2 = Program(H(q) for q in qbyte)
print(address_qubits(prog2, {q: i*2 for i, q in enumerate(qbyte)}))

H 0
H 2
H 4
H 6
H 8
H 10
H 12
H 14

